# Grocery & Produce Recognition Model Training

Run this notebook in Google Colab with GPU enabled (Runtime -> Change runtime type -> T4 GPU).

**Note:** We switched the dataset from `Food-101` (restaurant dishes) to a **Fruit & Vegetable Image Recognition** dataset. This dataset contains 36 common raw grocery items (like Apple, Banana, Carrot, Tomato, Onion, etc.), making it perfect for NutriLens!

In [ ]:
!pip install kaggle tensorflow scikit-learn

## Setup Kaggle API
Paste your `KAGGLE_API_TOKEN` (starts with `KGAT_...`) to download the datasets directly to Colab.

In [ ]:
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Enter your KAGGLE_API_TOKEN: ')

## Download Dataset

In [ ]:
!kaggle datasets download -d kritikseth/fruit-and-vegetable-image-recognition
!unzip -q fruit-and-vegetable-image-recognition.zip -d dataset
!ls dataset

## Train Model via Transfer Learning
We will use MobileNetV2 for fast and small inference footprint.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
import json

# Setup paths and data generators
train_dir = "dataset/train"
val_dir = "dataset/validation"

train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Build Model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_tensor=Input(shape=(224, 224, 3)))
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train Model
history = model.fit(train_generator, validation_data=val_generator, epochs=5)

# Save Model
model.save("food_recognition_model.h5")

# Save class mappings to JSON file
class_indices = train_generator.class_indices
classes_dict = {str(v): k for k, v in class_indices.items()}
with open('food_recognition_classes.json', 'w') as f:
    json.dump(classes_dict, f, indent=2)


## Download the Trained Model & Classes
After training, download both `food_recognition_model.h5` and `food_recognition_classes.json`. Place BOTH files into the `nutrilens-ai/ml_models` folder.

In [ ]:
from google.colab import files
files.download('food_recognition_model.h5')
files.download('food_recognition_classes.json')